<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim

# Word2Vec

In [ ]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt_tab')

# Example corpus
sentences = [
    "machine learning is fun",
    "word embeddings capture semantic meaning",
    "natural language processing uses embeddings",
    "deep learning models use word vectors"
]

# Tokenize
tokenized_sentences = [word_tokenize(sentence.lower()) for sentence in sentences]
print(tokenized_sentences)

# Train Word2Vec model
model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,   # embedding dimension
    window=5,          # context window
    min_count=1,       # ignore rare words
    workers=4,
    sg=1               # 1 = Skip-gram, 0 = CBOW
)

# Get embedding for a word
vector = model.wv['learning']
print(vector)

# Find similar words
similar = model.wv.most_similar('learning')
print(similar)

In [ ]:
from gensim.models import Word2Vec
import gensim
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download('punkt_tab')
import warnings
import zipfile
import requests

warnings.filterwarnings(action='ignore')

url = 'https://media.geeksforgeeks.org/wp-content/uploads/20250702123632592089/Gutenburg.zip'
filename = url.split('/')[-1]

response = requests.get(url)
if response.status_code == 200:
    with open(filename, 'wb') as f:
        f.write(response.content)
else:
    print("Failed to download file")

with zipfile.ZipFile("/content/Gutenburg.zip", 'r') as zip_ref:
    file_name = zip_ref.namelist()[0]
    with zip_ref.open(file_name) as file:
        content = file.read().decode('utf-8', errors='ignore')
        cleaned_text = content.replace("\n", " ")

data = []

for i in sent_tokenize(cleaned_text):
    temp = []

    # tokenize the sentence into words
    for j in word_tokenize(i):
        temp.append(j.lower())

    data.append(temp)

model1 = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5)
model2 = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5, sg=1)

print(f'Cosine similarity between \'alice\' and \'wonderland\' - CBOW: {model1.wv.similarity('alice', 'wonderland')}, Skip-Gram : {model2.wv.similarity('alice', 'wonderland')}')
print(f'Cosine similarity between \'alice\' and \'machines\' - CBOW: {model1.wv.similarity('alice', 'machines')}, Skip-Gram : {model2.wv.similarity('alice', 'machines')}')

# GloVe: Global Vectors for Word Representation

In [ ]:
!wget https://nlp.stanford.edu/data/wordvecs/glove.2024.dolma.300d.zip
!unzip glove.2024.dolma.300d.zip

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors

In [ ]:
def load_glove_embeddings(file_path):
    embeddings_dict = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings_dict[word] = vector
    return embeddings_dict

# Example usage:
glove_file = "/content/dolma_300_2024_1.2M.100_combined.txt" # Make sure to use the correct file path
embeddings_dict = load_glove_embeddings(glove_file)

In [ ]:
embeddings_dict['king'].reshape(1, -1).shape

In [ ]:
w_embed1 = embeddings_dict['king'].reshape(1, -1)
w_embed2 = embeddings_dict['queen'].reshape(1, -1)
cosine_similarity(w_embed1, w_embed2)

In [ ]:
# DO NOT RUN. Takes too long.
def find_similar_words(target_word, model, top_n=5):
    if target_word not in model:
        return []

    target_vec = model[target_word].reshape(1, -1)

    # Calculate cosine similarity
    similarities = {}
    for word, vec in model.items():
        if word == target_word: continue
        similarities[word] = cosine_similarity(target_vec, vec.reshape(1, -1))[0][0]

    # Sort by similarity
    sorted_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return sorted_words[:top_n]

# Example: Similar to "king"
# print(find_similar_words("king", embeddings_dict))

In [ ]:
# DO NOT RUN. Takes too long.
# Convert and Load
# glove2word2vec('/content/dolma_300_2024_1.2M.100_combined.txt', '/content/dolma_300_2024_1.2M.100_combined.wv')
# model = KeyedVectors.load_word2vec_format('/content/dolma_300_2024_1.2M.100_combined.wv', binary=False)

# Find similar
# print(model.most_similar("king"))

# BERT (Bi-direction Encoder Representations from Transformers)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

In [ ]:
#@title BERT - Masked Language Model (MLM) using Transformers Pipeline
pipe = pipeline("fill-mask", model="google-bert/bert-base-uncased")
pipe('The capital of India is [MASK].')

In [ ]:
#@title BERT - Masked Language Model (MLM) using PyTorch
import torch
from transformers import BertTokenizer, BertForMaskedLM

# load pretrained model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# sentence with mask
text = "Paris is the [MASK] of France."

# tokenize
inputs = tokenizer(text, return_tensors="pt")

# run model
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

# find mask token index
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# get probabilities
mask_token_logits = logits[0, mask_token_index, :]

top_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

print("Predictions:")
for token in top_tokens:
    print(text.replace("[MASK]", tokenizer.decode([token])))

In [ ]:
#@title BERT - Next Sentence Prediction (NSP) using PyTorch
import torch
from transformers import BertTokenizer, BertForNextSentencePrediction

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased")

sentence_A = "Paris is the capital of France."
sentence_B = "It is known for the Eiffel Tower."
sentence_C = "I am hungry."

# Tokenize pair of sentences
encoding1 = tokenizer(sentence_A, sentence_B, return_tensors="pt")
encoding2 = tokenizer(sentence_A, sentence_C, return_tensors="pt")

# Run model
with torch.no_grad():
    outputs1 = model(**encoding1)
    outputs2 = model(**encoding2)

logits1 = outputs1.logits
probs1 = torch.softmax(logits1, dim=1)
logits2 = outputs2.logits
probs2 = torch.softmax(logits2, dim=1)

print(f"Sentence Pair: (A:{sentence_A}, B:{sentence_B}), IsNext probability: {probs1[0][0].item()}, NotNext probability: {probs1[0][1].item()}")
print(f"Sentence Pair: (A:{sentence_A}, B:{sentence_C}), IsNext probability: {probs2[0][0].item()}, NotNext probability: {probs2[0][1].item()}")